In [1]:
import pandas as pd
import numpy as np
import requests
import time
from itertools import groupby
import pyarrow

In [21]:
def get_schedule(seasons: list, start: str, end: str) -> pd.DataFrame:

    """
    Fetches the MLB schedule for the specified date range and returns a DataFrame with game details.
    Args:
        seasons (list): A list of seasons for which to fetch schedule data.
        start (str): The start date in 'YYYY-MM-DD' format.
        end (str): The end date in 'YYYY-MM-DD' format.
    Returns:
        pd.DataFrame: A DataFrame containing gamePk, date, homeTeam, and awayTeam for each game in the specified date range.
    """

    start = pd.to_datetime(start, utc=True)
    end = pd.to_datetime(end, utc=True)

    schedule_df = pd.DataFrame()

    for season in seasons:
  
        schedule_data = requests.get(f'https://statsapi.mlb.com/api/v1/schedule?sportId=1&season={season}').json()

        games = [
            game
            for date in schedule_data.get('dates', [])
            for game in date.get('games', [])
            if game.get('gameType') == 'R'
            and game.get('status', {}).get('abstractGameState') == 'Final'
            and game.get('status', {}).get('detailedState') not in ('Postponed', 'Cancelled')
        ]


        season_df = pd.DataFrame()
        season_df['gamePk'] = [game['gamePk'] for game in games]
        season_df['date'] = pd.to_datetime([game['gameDate'] for game in games], utc=True)
        season_df['homeTeam'] = [game['teams']['home']['team']['name'] for game in games]
        season_df['awayTeam'] = [game['teams']['away']['team']['name'] for game in games]

        schedule_df = pd.concat([schedule_df, season_df], ignore_index=True)

    schedule_df = schedule_df[(schedule_df['date'] >= pd.to_datetime(start)) & (schedule_df['date'] <= pd.to_datetime(end))].drop_duplicates('gamePk', keep='last').reset_index(drop=True)

    print(f'Total games: {len(schedule_df)}')

    return schedule_df

In [22]:
schedule = get_schedule(seasons=[2026], start='2026-03-25', end='2026-09-28')

Total games: 2429


In [7]:
def drop_incomplete_abs(df):
    pitch_data_cols = ['startSpeed', 'pX', 'pZ', 'strikeZoneTop', 'strikeZoneBottom']
    incomplete_abs = (
        df[df[pitch_data_cols].isnull().any(axis=1)][['gamePk', 'atBatIndex']]
        .drop_duplicates()
    )
    n = len(incomplete_abs)
    df_clean = df.merge(incomplete_abs, on=['gamePk', 'atBatIndex'], how='left', indicator=True)
    df_clean = df_clean[df_clean['_merge'] == 'left_only'].drop(columns='_merge')
    print(f"Dropped {n} incomplete at bats ({len(df) - len(df_clean)} pitches removed)")
    return df_clean.reset_index(drop=True)

In [8]:
def midpoint(df, pY_mid=(8.5/12)):

    """
    Convert MLB pitch coordinates (front of plate) to ABS pitch coordinates (plate mid point)
    Requires df containing statcast pitch data points x0, y0, z0, vX0, vY0, vZ0, aX, aY, aZ
    returns df with new columns: 
        - pX_mid (horizontal location of the pitch at the midpoint), 
        - pZ_mid (vertical location of the pitch at the midpoint),
        - pY_mid (depth of the pitch at the midpoint, default 8.5 inches converted to feet), 
        - t (time from front to midpoint)
    """

    a = 0.5 * df['aY']
    b = df['vY0']
    c = df['y0'] - pY_mid
    disc = b**2 - 4*a*c

    # smaller positive root: -b is positive (vY0 < 0), so subtract the sqrt
    t = (-b - np.sqrt(disc)) / (2*a)
    pX_mid = df['x0'] + df['vX0']*t + 0.5*df['aX']*t**2
    pZ_mid = df['z0'] + df['vZ0']*t + 0.5*df['aZ']*t**2

    df['pX_mid'] = pX_mid
    df['pZ_mid'] = pZ_mid
    df['pY_mid'] = pY_mid
    df['plate_time'] = t

    return df

In [9]:
def original_call(df):

    df['originalCallCode'] = np.where(
        (df['isOverturned'] == True) & (df['call_code'] == 'B') & (df['reviewType'] == 'MJ'),
        'C',
        np.where(
            (df['isOverturned'] == True) & (df['call_code'] == 'C') & (df['reviewType'] == 'MJ'),
            'B',
            df['call_code']  # not overturned: original == final
        )
    )
    
    code_to_call = {'C': 'Called Strike', 'B': 'Ball'}
    df['originalCall'] = df['originalCallCode'].map(code_to_call)

    return df

In [ ]:
def signed_edge_dist(df):
    """
    Calculate signed distance from the edge of the strike zone at the midpoint of the pitch
    Requires df containing pX_mid, pZ_mid, strikeZoneTop, strikeZoneBottom
    returns df with new columns 'off_edge_dist' (positive outside, hitter's perspective) 
    and 'def_edge_dist' (positive inside, defensive perspective)
    """

    x_left = -0.7083 - .122
    x_right = 0.7083 + .122
    z_top = df['strikeZoneTop'] + .122
    z_bottom = df['strikeZoneBottom'] - .122
    px = df['pX_mid']
    pz = df['pZ_mid']

    dx = np.maximum(np.maximum(x_left - px, 0), px - x_right)
    dz = np.maximum(np.maximum(z_bottom - pz, 0), pz - z_top)

    outside_dist = np.sqrt(dx**2 + dz**2)
    inside_dist = np.minimum(np.minimum(np.minimum(px - x_left, x_right - px), pz - z_bottom), z_top - pz)


    df['off_edge_dist'] = np.where(outside_dist > 0, outside_dist, -inside_dist)
    df['def_edge_dist'] = -df['off_edge_dist']

    return df

In [11]:
def abs_classifier(df):
    """
    Classify pitches as TP, FP, FN, TN based on the following criteria:
    - TP: hasReview == True and overturned == True
    - FP: hasReview == True and overturned == False
    - FN: 
        - call == 'Called Strike' and (abs(pX_mid) > 8.5/12 or pZ_mid < (strikeZoneBottom - 0.122) or pZ_mid > (strikeZoneTop + 0.122))
        OR
        - call == 'Ball' and abs(pX_mid) < 8.5/12 and pZ_mid > (strikeZoneBottom - 0.122) and pZ_mid < (strikeZoneTop + 0.122)
    - TN: not TP, not FP, not FN
    """
        
    df['TP'] = ((df['reviewType'] == 'MJ') & (df['isOverturned'] == True))

    df['FP'] = ((df['reviewType'] == 'MJ') & (df['isOverturned'] == False))

    df['Off_FN'] = (
        ~(df['reviewType'] == 'MJ') &
        (df['call'] == 'Called Strike') &
        (
            (abs(df['pX_mid']) > 8.5/12 + 0.122) |
            (df['pZ_mid'] < df['strikeZoneBottom'] - 0.122) |
            (df['pZ_mid'] > df['strikeZoneTop'] + 0.122)
        )
    )

    df['Def_FN']= (
        ~(df['reviewType'] == 'MJ') &
        (df['call'] == 'Ball') &
        (
            (abs(df['pX_mid']) < 8.5/12 + 0.122) &
            (df['pZ_mid'] > df['strikeZoneBottom'] - 0.122) &
            (df['pZ_mid'] < df['strikeZoneTop'] + 0.122)
        )
    )

    df['TN'] = ~(df['TP'] | df['Off_FN'] | df['Def_FN'] | df['FP'])

    return df

In [12]:
def half_inning_key(play: dict) -> tuple:
    about = play.get('about', {})
    return (about.get('inning'), about.get('halfInning'))


def get_runner_states(data: dict) -> tuple[dict, dict, dict, dict, dict]:
    """
    Reconstructs base-runner occupancy, out count, and score across an entire
    game. Requires the full game feed (data['liveData']['plays']['allPlays']).

    Tracks runner state as {runner_id: base}, persisted across plays and reset
    only at half-inning boundaries — runners absent from a play's 'runners'
    array are still on base and must not be dropped. Out count is tracked the
    same way, using each out's 'outNumber' (MLB's own running half-inning out
    counter, present on every isOut runner movement, including batter outs —
    validated against the raw feed's own pre-pitch outs on the following play).

    Score is tracked the same way too, via the same runner movements: a
    movement with `end == 'score'` is a run, credited to the batting team of
    that half-inning. This reuses the exact signal that already identifies a
    scoring runner for base_state removal — no separate mechanism needed.
    The event-level `details.homeScore`/`awayScore` field was considered and
    rejected as the source: it's `None` on essentially every `type: 'pitch'`
    event and only sporadically populated on some `action` events, so it
    can't give a reliable pre-pitch value on every row. `play.result
    .homeScore`/`awayScore` is reliable but only available at the end of a
    whole play — no good for a run that scores mid-play with more pitches to
    the same batter still to come (e.g. a run on a wild pitch before the at-
    bat concludes). The runner-movement signal has neither gap.

    Two additional wrinkles, both confirmed against real 2026 data:
    - Pinch-runner substitutions swap who occupies a base but have **no**
      entry in the `runners` movement array — they're a separate `playEvent`
      (`isSubstitution=True`, `position.abbreviation == 'PR'`) carrying
      `base` (int), `player.id` (incoming), `replacedPlayer.id` (outgoing).
      Handled inline by re-keying base_state at the matching event index.
    - Extra innings (10th on) place an automatic runner on 2B via an explicit,
      structured `playEvent` — `details.eventType == 'runner_placed'`,
      carrying `base` (int) and `player.id` directly — present on all 20
      extra-inning half-innings in a 76-game validation sample. This
      supersedes an earlier heuristic (inferring the placed runner from
      batting-order continuity, i.e. the last batter of the team's previous
      half-inning) that broke whenever a substitution meant the placed runner
      wasn't literally the same person who made that last out — the explicit
      event needs no inference at all and is handled the same way as PR
      substitutions, inline in the event loop.

    `pre_pitch`/`pre_score` are recorded for **every** playEvent, not only
    pitches, so pre-substitution/pre-pickoff state is available for
    validation at any event index — extract_pitch only ever looks up
    pitch-type indices, so this is a pure superset with no behavior change
    for pitch rows.

    Returns:
        pre_pitch: {(atBatIndex, eventIndex): {runner_id: base}}, snapshot
            taken before that event's own movements are applied.
        post_play: {atBatIndex: {runner_id: base}}, snapshot taken after all
            of a play's movements are applied.
        post_outs: {atBatIndex: int}, outs after all of a play's movements
            are applied.
        pre_score: {(atBatIndex, eventIndex): {team_id: score}}, snapshot
            taken before that event's own movements are applied.
        final_score: {team_id: score}, for validation against the linescore.
    """
    pre_pitch = {}
    post_play = {}
    post_outs = {}
    pre_score = {}
    base_by_number = {1: '1B', 2: '2B', 3: '3B'}
    home_id = data['gameData']['teams']['home']['id']
    away_id = data['gameData']['teams']['away']['id']
    score = {home_id: 0, away_id: 0}

    for half_key, plays_iter in groupby(data['liveData']['plays']['allPlays'], key=half_inning_key):
        base_state = {}
        outs_count = 0
        batting_team_id = home_id if half_key[1] == 'bottom' else away_id

        for play in plays_iter:
            about = play.get('about', {})
            ab_idx = about.get('atBatIndex')

            movements_by_index = {}
            out_numbers = []
            for runner in play.get('runners', []):
                details = runner['details']
                play_index = details.get('playIndex')
                runner_id = details.get('runner', {}).get('id')
                movement = runner['movement']
                if play_index is not None:
                    movements_by_index.setdefault(play_index, []).append((runner_id, movement))
                if movement.get('isOut') and movement.get('outNumber') is not None:
                    out_numbers.append(movement['outNumber'])

            for event in play.get('playEvents', []):
                idx = event['index']

                pre_pitch[(ab_idx, idx)] = dict(base_state)
                pre_score[(ab_idx, idx)] = dict(score)

                # Sequential (not batched) apply — required for same-runner multi-hop
                # moves within one event index (e.g. None->1B then 1B->2B).
                for runner_id, movement in movements_by_index.get(idx, []):
                    base_state.pop(runner_id, None)
                    end = movement.get('end')
                    is_out = movement.get('isOut', False)
                    if end == 'score':
                        score[batting_team_id] += 1
                    elif end is not None and not is_out:
                        base_state[runner_id] = end

                # Pinch-runner substitutions swap who occupies a base without a
                # corresponding entry in the runners/movement array.
                if event.get('isSubstitution') and event.get('position', {}).get('abbreviation') == 'PR':
                    base = base_by_number.get(event.get('base'))
                    old_runner_id = event.get('replacedPlayer', {}).get('id')
                    new_runner_id = event.get('player', {}).get('id')
                    if base is not None and base_state.get(old_runner_id) == base:
                        del base_state[old_runner_id]
                        base_state[new_runner_id] = base

                # Extra-innings automatic runner: explicit placement event.
                if event.get('details', {}).get('eventType') == 'runner_placed':
                    base = base_by_number.get(event.get('base'))
                    placed_runner_id = event.get('player', {}).get('id')
                    if base is not None and placed_runner_id is not None:
                        base_state[placed_runner_id] = base

            if out_numbers:
                outs_count = max(out_numbers)

            post_play[ab_idx] = dict(base_state)
            post_outs[ab_idx] = outs_count

    return pre_pitch, post_play, post_outs, pre_score, dict(score)


def format_base_state(base_state: dict) -> str:
    return ', '.join(sorted(base_state.values())) if base_state else '0'


def validate_runner_states(data: dict, pre_pitch: dict, post_play: dict, post_outs: dict) -> list[str]:
    """
    Three consistency checks over a single game's runner/out states:
    1. Each runner's *first* movement within a play must start from a base
       already reflected in the pre-event state at that movement's own event
       index — not necessarily the play's first pitch, since pinch-runner
       substitutions can change who occupies a base mid-play before a
       runner's own movement is recorded.
    2. post_play[n] must equal the pre-event state at play n+1's first event
       (not necessarily its first *pitch* — a leading substitution or pickoff
       can occur before the first pitch, within the same play), within the
       same half-inning (half-inning boundaries reset state instead).
    3. post_outs[n] must equal the raw feed's own outs at play n+1's first
       event (same reasoning as #2 — e.g. a pickoff out before the first
       pitch of the next batter's plate appearance), same half-inning.
    Returns a list of error message strings (empty if all checks pass).
    """
    errors = []
    prev_ab_idx = None
    prev_half_key = None

    for play in data['liveData']['plays']['allPlays']:
        about = play.get('about', {})
        ab_idx = about.get('atBatIndex')
        half_key = (about.get('inning'), about.get('halfInning'))

        events = play.get('playEvents', [])
        first_event = events[0] if events else None
        first_event_key = (ab_idx, first_event['index']) if first_event else None
        first_event_state = pre_pitch.get(first_event_key) if first_event_key else None
        first_event_outs = first_event.get('count', {}).get('outs') if first_event else None

        first_movement = {}
        for runner in play.get('runners', []):
            details = runner['details']
            runner_id = details.get('runner', {}).get('id')
            play_index = details.get('playIndex')
            if play_index is None or runner_id is None:
                continue
            if runner_id not in first_movement or play_index < first_movement[runner_id][0]:
                first_movement[runner_id] = (play_index, runner['movement'])

        for runner_id, (play_index, movement) in first_movement.items():
            start = movement.get('start')
            state_at_movement = pre_pitch.get((ab_idx, play_index), {})
            if start is not None and state_at_movement.get(runner_id) != start:
                errors.append(
                    f"atBatIndex={ab_idx} runner={runner_id}: first movement start={start!r} "
                    f"not reflected in pre-event state at idx={play_index} ({state_at_movement})"
                )

        if prev_ab_idx is not None and half_key == prev_half_key:
            if first_event_key is not None and post_play.get(prev_ab_idx) != first_event_state:
                errors.append(
                    f"halfInning continuity: post_play[{prev_ab_idx}]={post_play.get(prev_ab_idx)} "
                    f"!= pre_pitch[{first_event_key}]={first_event_state}"
                )
            if first_event_outs is not None and post_outs.get(prev_ab_idx) != first_event_outs:
                errors.append(
                    f"halfInning continuity: post_outs[{prev_ab_idx}]={post_outs.get(prev_ab_idx)} "
                    f"!= raw outs at atBatIndex={ab_idx} first event ({first_event_outs})"
                )

        prev_ab_idx = ab_idx
        prev_half_key = half_key

    return errors


def validate_score_states(data: dict, final_score: dict) -> list[str]:
    """
    Final reconstructed score must match the linescore's own final runs per
    team — ground truth independent of our own runner-movement reconstruction.
    Returns a list of error message strings (empty if all checks pass).
    """
    errors = []
    home_id = data['gameData']['teams']['home']['id']
    away_id = data['gameData']['teams']['away']['id']
    ls_teams = data.get('liveData', {}).get('linescore', {}).get('teams', {})
    expected = {
        home_id: ls_teams.get('home', {}).get('runs'),
        away_id: ls_teams.get('away', {}).get('runs'),
    }

    for team_id, expected_runs in expected.items():
        if expected_runs is not None and final_score.get(team_id) != expected_runs:
            errors.append(
                f"team={team_id}: reconstructed final score={final_score.get(team_id)} "
                f"!= linescore runs={expected_runs}"
            )

    return errors

In [13]:
def apply_call_effect(balls: int, strikes: int, code: str) -> tuple[int, int]:
    """
    Applies one pitch's count effect (call_code is already the effective,
    post-overturn call — confirmed against the raw feed: an overturned called
    strike shows call_code='B' with the count already reflecting a ball, not
    the original human call). O is a 'Foul Tip' (18 rows, confirmed identical
    in behavior to T): unconditional +1 strike, not a capped true foul.

    2026 pitch-clock rules add non-pitch events that still change the count —
    confirmed against real data:
    - `VB` (Automatic Ball - Intentional, the no-pitch IBB shortcut), `VP`
      (Automatic Ball - Pitcher Pitch Timer Violation), `VC` (Automatic Ball
      - Catcher Pitch Timer Violation), and `V` (generic Automatic Ball, no
      violation subtype given) all behave like B.
    - `AC` (Automatic Strike - Batter Pitch Timer Violation) behaves like C.
    - `P` (Pitchout) is a real `type: 'pitch'` event (unlike the V*/AC codes)
      that behaves like a ball — confirmed: pre-count 1 ball -> post 2 balls.
    - `Q` is a swinging strike at a pitchout — a strike per MLB rules (batter
      swung, so the pitch being a pitchout is irrelevant to the call).
    - `H` (Hit By Pitch) is scored as a ball in the raw feed (`isBall: True`),
      even though the plate appearance ends via HBP.
    - `M` (Missed Bunt) is a swing-and-miss: unconditional +1 strike.
    - `L` (Foul Bunt) is **not** capped like a regular foul — rule 5.09(a)(4):
      a foul bunt with two strikes is strike three. Unconditional +1 strike,
      grouped with the swinging-strike-like codes, not with `F`.
    """
    if code in ('B', '*B', 'H', 'VB', 'VP', 'VC', 'V', 'P'):
        return balls + 1, strikes
    if code in ('C', 'S', 'W', 'T', 'O', 'AC', 'M', 'L', 'Q', 'AB'):
        return balls, strikes + 1
    if code == 'F' and strikes < 2:
        return balls, strikes + 1
    return balls, strikes


def get_count_states(data: dict) -> dict:
    """
    Reconstructs pre-pitch balls/strikes for every pitch in a game, built
    forward from 0-0 at the start of each at-bat. Count effects are applied
    for **every** play event that carries a call code, not only
    `type == 'pitch'` ones — pitch-clock automatic balls/strikes (VB, VP, VC,
    V, AC) are recorded as separate non-pitch events that still change the
    count.

    Returns:
        pre_count: {(atBatIndex, eventIndex): (balls, strikes)}, snapshot
            taken before that pitch's own call is applied.
    """
    pre_count = {}

    for play in data['liveData']['plays']['allPlays']:
        ab_idx = play.get('about', {}).get('atBatIndex')
        balls, strikes = 0, 0

        for event in play.get('playEvents', []):
            if event.get('type') == 'pitch':
                pre_count[(ab_idx, event['index'])] = (balls, strikes)
            code = event.get('details', {}).get('call', {}).get('code')
            balls, strikes = apply_call_effect(balls, strikes, code)

    return pre_count


def validate_count_states(data: dict, pre_count: dict) -> list[str]:
    """
    Recomputed post-pitch balls/strikes (pre_count + effect of this pitch's
    call_code) must match the raw recorded balls/strikes on every pitch.
    Returns a list of error message strings (empty if all checks pass).
    """
    errors = []
    for play in data['liveData']['plays']['allPlays']:
        ab_idx = play.get('about', {}).get('atBatIndex')
        for event in play.get('playEvents', []):
            if event.get('type') != 'pitch':
                continue
            idx = event['index']
            code = event.get('details', {}).get('call', {}).get('code')
            balls, strikes = apply_call_effect(*pre_count[(ab_idx, idx)], code)
            raw = event.get('count', {})
            if raw.get('balls') != balls or raw.get('strikes') != strikes:
                errors.append(
                    f"atBatIndex={ab_idx} eventIndex={idx} code={code!r}: "
                    f"reconstructed post-pitch ({balls},{strikes}) != raw ({raw.get('balls')},{raw.get('strikes')})"
                )
    return errors

In [14]:
def get_catcher_states(data: dict) -> dict[tuple[int, int], int]:
    """
    Returns a mapping of (atBatIndex, eventIndex) -> catcher player_id for
    every pitch. Accounts for mid-game catcher substitutions, including ones
    made mid-at-bat (hence keyed per pitch, not per at-bat).

    The starting catcher is the non-substitute whose *first* listed position
    is C. `allPositions` lists every position a player held in the game, in
    order, so a starter who began at DH and later moved behind the plate
    (e.g. ['DH', 'C']) also contains 'C' — matching on 'C' anywhere in the
    list credited his DH innings to him instead of the actual starting
    catcher (found via ABS challengers who were neither batter, pitcher, nor
    assigned catcher: 33 in a 101-game sample, 0 after this fix).
    """
    boxscore_teams = data['liveData']['boxscore']['teams']
    game_teams = data['gameData']['teams']

    home_team_id = game_teams['home']['id']
    away_team_id = game_teams['away']['id']

    def find_starting_catcher(side: str) -> int:
        for p in boxscore_teams[side]['players'].values():
            positions = [pos['abbreviation'] for pos in p.get('allPositions', [])]
            if positions[:1] == ['C'] and not p.get('gameStatus', {}).get('isSubstitute', False):
                return p['person']['id']
        return None

    # { team_id -> catcher_id }, initialized with starters
    active_catchers = {
        home_team_id: find_starting_catcher('home'),
        away_team_id: find_starting_catcher('away'),
    }

    # The fielding team is opposite the batting team
    # top = away batting, home fielding; bottom = home batting, away fielding
    def fielding_team_id(half_inning: str) -> int:
        return home_team_id if half_inning == 'top' else away_team_id

    all_plays = data['liveData']['plays']['allPlays']
    result = {}

    for play in all_plays:
        ab_idx = play.get('about', {}).get('atBatIndex')
        half = play.get('about', {}).get('halfInning')
        fielding_team = fielding_team_id(half)

        for event in play.get('playEvents', []):
            # Process catcher subs before recording pitch state
            if (event.get('isSubstitution') and
                    event.get('position', {}).get('abbreviation') == 'C'):
                active_catchers[fielding_team] = event['player']['id']

            if event.get('type') == 'pitch':
                result[(ab_idx, event['index'])] = active_catchers[fielding_team]

    return result


def validate_catcher_states(data: dict, catcher_states: dict) -> list[str]:
    """
    Two checks per pitch:
    1. The assigned catcher is on the fielding team's boxscore roster.
    2. Any ABS challenge (MJ) made by someone other than the batter or pitcher
       was made by the assigned catcher.
    Returns a list of error message strings (empty if all checks pass).
    """
    errors = []
    boxscore_teams = data['liveData']['boxscore']['teams']
    rosters = {
        side: {p['person']['id'] for p in boxscore_teams[side]['players'].values()}
        for side in ('home', 'away')
    }

    for play in data['liveData']['plays']['allPlays']:
        about = play.get('about', {})
        ab_idx = about.get('atBatIndex')
        fielding_side = 'home' if about.get('halfInning') == 'top' else 'away'
        matchup = play.get('matchup', {})
        batter_pitcher = {matchup.get('batter', {}).get('id'), matchup.get('pitcher', {}).get('id')}
        pitch_index = play.get('pitchIndex', [])

        for event in play.get('playEvents', []):
            if event.get('type') != 'pitch':
                continue
            idx = event['index']
            catcher_id = catcher_states.get((ab_idx, idx))
            if catcher_id not in rosters[fielding_side]:
                errors.append(f"atBatIndex={ab_idx} eventIndex={idx}: catcher={catcher_id} not on fielding roster")

            is_last_pitch = bool(pitch_index) and idx == pitch_index[-1]
            review = event.get('reviewDetails') or (play.get('reviewDetails') if is_last_pitch else None) or {}
            challenger = review.get('player', {}).get('id')
            if review.get('reviewType') == 'MJ' and challenger not in batter_pitcher and challenger != catcher_id:
                errors.append(
                    f"atBatIndex={ab_idx} eventIndex={idx}: challenger={challenger} "
                    f"is not batter, pitcher, or assigned catcher={catcher_id}"
                )

    return errors

In [15]:
def get_challenge_states(data: dict) -> tuple[dict, dict]:
    """
    Reconstructs each team's remaining ABS ball/strike challenges before
    every event. Starts at 2 per team, decremented (floored at 0) only on a
    FAILED challenge — retained on success, per the ABS rule.

    Only `reviewType == 'MJ'` counts as an ABS challenge. MLB's feed also
    carries `MA`/`MB` review types on the same games — a separate,
    pre-existing instant-replay system (tag plays, boundary calls, etc.)
    tracked independently in `gameData.review`, not `gameData.absChallenges`.
    Confirmed by counting each type separately on a real game: `MJ` counts
    exactly match `gameData.absChallenges`' `usedSuccessful`/`usedFailed`;
    `MA`/`MB` counts exactly match `gameData.review`'s `used` counts.

    A play can carry **more than one** `MJ` review — a team can challenge
    more than once within the same plate appearance, on different pitches
    (confirmed on a real at-bat: two consecutive-pitch challenges, both by
    the same team, both successful — Eugenio Suárez, CIN@BOS 2026-03-28,
    27 such multi-review plays found in a 160-game sample). Each event's own
    reviewDetails is checked, and its effect applied **immediately after
    that event's own pre-state snapshot** — not deferred to the end of the
    play — so a second challenge later in the same at-bat correctly sees the
    outcome of an earlier one in its own pre-state.

    The play-level review (reviewDetails on the play itself, not a specific
    event — used only when the challenge ends the at-bat) is applied after
    the event loop; there's no later event within the same play whose
    pre-state could be affected, since the at-bat is already over.

    **Extra innings (10th on) floor each team's challenges at 1 — once, not
    accumulating.** A team entering the 10th with 0 remaining is bumped up to
    1; a team with 1 or 2 already is left alone (they only ever reach 2 by
    entering extras with both original challenges intact). This is a
    one-time adjustment at the first half-inning with `inning >= 10`, not
    reapplied in the 11th/12th/etc. — confirmed against all 117 extra-inning
    games in a full-season sample: the "excess" (usedFailed beyond the base
    2) never exceeds 1 regardless of how many extra innings are played,
    which rules out a per-inning grant.

    This floor deliberately diverges from `gameData.absChallenges.remaining`
    for a team that receives the floor bump but never spends it: MLB's own
    field is a simplified `max(0, 2 - usedFailed)` display stat that doesn't
    reflect the unused bonus (confirmed: of 234 team-instances across all
    117 extra-inning games this season, every single mismatch against that
    field is exactly this "+1, unused bonus" case — zero unexplained
    mismatches). We track true spendable availability instead, since that's
    what the option-value framework needs (an unused bonus challenge still
    has option value) — see `validate_challenge_states` for how this
    expected divergence is distinguished from an actual bug.

    Returns:
        pre_pitch_challenges: {(atBatIndex, eventIndex): {team_id: remaining}},
            snapshot taken before that event's own review (if any) is applied.
        final_remaining: {team_id: remaining}, for validation against
            gameData.absChallenges.
    """
    home_id = data['gameData']['teams']['home']['id']
    away_id = data['gameData']['teams']['away']['id']
    remaining = {home_id: 2, away_id: 2}
    pre_pitch_challenges = {}
    extra_innings_floor_applied = False

    for play in data['liveData']['plays']['allPlays']:
        about = play.get('about', {})
        ab_idx = about.get('atBatIndex')
        inning = about.get('inning')

        if inning is not None and inning >= 10 and not extra_innings_floor_applied:
            remaining[home_id] = max(remaining[home_id], 1)
            remaining[away_id] = max(remaining[away_id], 1)
            extra_innings_floor_applied = True

        for event in play.get('playEvents', []):
            idx = event['index']
            pre_pitch_challenges[(ab_idx, idx)] = dict(remaining)

            rd = event.get('reviewDetails')
            if rd and rd.get('reviewType') == 'MJ' and rd.get('isOverturned') is False:
                team_id = rd.get('challengeTeamId')
                if team_id in remaining:
                    remaining[team_id] = max(0, remaining[team_id] - 1)

        prd = play.get('reviewDetails')
        if prd and prd.get('reviewType') == 'MJ' and prd.get('isOverturned') is False:
            team_id = prd.get('challengeTeamId')
            if team_id in remaining:
                remaining[team_id] = max(0, remaining[team_id] - 1)

    return pre_pitch_challenges, dict(remaining)


def validate_challenge_states(data: dict, final_remaining: dict) -> list[str]:
    """
    Final reconstructed remaining challenges must match gameData.absChallenges'
    own remaining counts — ground truth computed by MLB itself — with one
    known, expected exception: a team that received the extra-innings floor
    bump (see get_challenge_states) but never spent it will show 1 more than
    gameData.absChallenges reports, since that field doesn't account for an
    unused bonus. That specific case (reconstructed = expected + 1, and
    expected == 0) is not flagged; anything else is.
    Returns a list of error message strings (empty if all checks pass).
    """
    errors = []
    abs_challenges = data.get('gameData', {}).get('absChallenges', {})
    if not abs_challenges.get('hasChallenges'):
        return errors

    home_id = data['gameData']['teams']['home']['id']
    away_id = data['gameData']['teams']['away']['id']
    expected = {
        home_id: abs_challenges.get('home', {}).get('remaining'),
        away_id: abs_challenges.get('away', {}).get('remaining'),
    }

    for team_id, expected_remaining in expected.items():
        if expected_remaining is None:
            continue
        recon = final_remaining.get(team_id)
        if recon == expected_remaining:
            continue
        if recon == expected_remaining + 1 and expected_remaining == 0:
            continue  # unused extra-innings floor bump — expected divergence
        errors.append(
            f"team={team_id}: reconstructed final remaining={recon} "
            f"!= gameData.absChallenges remaining={expected_remaining}"
        )

    return errors

In [16]:
def extract_pitch(event: dict, play: dict, data: dict, pre_pitch: dict, pre_count: dict, post_outs: dict, pre_challenges: dict, pre_score: dict, catcher_states: dict) -> dict:
    result = play.get('result', {})
    about = play.get('about', {})
    matchup = play.get('matchup', {})
    details = event.get('details', {})
    count = event.get('count', {})
    pitchData = event.get('pitchData', {})
    coords = pitchData.get('coordinates', {})
    pitch_index = play.get('pitchIndex', [])
    is_last_pitch = bool(pitch_index) and event.get('index') == pitch_index[-1]
    review = event.get('reviewDetails') or (
        play.get('reviewDetails') if is_last_pitch else None
    ) or {}
    players = data.get('gameData', {}).get('players', {})
    ab_idx = about.get('atBatIndex')
    pre_balls, pre_strikes = pre_count.get((ab_idx, event['index']), (None, None))
    catcher_id = catcher_states.get((ab_idx, event['index']))

    home_team = data['gameData']['teams']['home']
    away_team = data['gameData']['teams']['away']
    is_home_batting = about.get('halfInning') == 'bottom'
    batting_team = home_team if is_home_batting else away_team
    fielding_team = away_team if is_home_batting else home_team
    team_challenges = pre_challenges.get((ab_idx, event['index']), {})
    team_score = pre_score.get((ab_idx, event['index']), {})

    return {
        'date': data.get('gameData', {}).get('datetime', {}).get('officialDate'),
        'gamePk': data.get('gamePk'),
        'atBatIndex': ab_idx,
        'eventIndex': event.get('index'),
        'halfInning': about.get('halfInning'),
        'inning': about.get('inning'),
        'homeTeam': home_team.get('abbreviation'),
        'awayTeam': away_team.get('abbreviation'),
        'battingTeam': batting_team.get('abbreviation'),
        'fieldingTeam': fielding_team.get('abbreviation'),
        'batter_id': matchup.get('batter', {}).get('id'),
        'batter': matchup.get('batter', {}).get('fullName'),
        'batSide': matchup.get('batSide', {}).get('code'),
        'pitcher_id': matchup.get('pitcher', {}).get('id'),
        'pitcher': matchup.get('pitcher', {}).get('fullName'),
        'pitchHand': matchup.get('pitchHand', {}).get('code'),
        'catcher_id': catcher_id,
        'catcher': players.get(f'ID{catcher_id}', {}).get('fullName') if catcher_id else None,
        'call_code': details.get('call', {}).get('code'),
        'call': details.get('call', {}).get('description'),
        'hasReview': bool(review),
        'reviewType': review.get('reviewType'),
        'challengeTeamId': review.get('challengeTeamId'),
        'isOverturned': review.get('isOverturned'),
        'review_player_name': review.get('player', {}).get('fullName'),
        'review_player_id': review.get('player', {}).get('id'),
        'pre_balls': pre_balls,
        'pre_strikes': pre_strikes,
        'pre_outs': count.get('outs'),
        'post_outs': post_outs.get(ab_idx),
        'runners': format_base_state(pre_pitch.get((ab_idx, event['index']), {})),
        'pre_batting_challenges': team_challenges.get(batting_team['id']),
        'pre_fielding_challenges': team_challenges.get(fielding_team['id']),
        'pre_home_score': team_score.get(home_team['id']),
        'pre_away_score': team_score.get(away_team['id']),
        'startSpeed': pitchData.get('startSpeed'),
        'endSpeed': pitchData.get('endSpeed'),
        'strikeZoneTop': pitchData.get('strikeZoneTop'),
        'strikeZoneBottom': pitchData.get('strikeZoneBottom'),
        'strikeZoneWidth': pitchData.get('strikeZoneWidth'),
        'strikeZoneDepth': pitchData.get('strikeZoneDepth'),
        'aY': coords.get('aY'),
        'aZ': coords.get('aZ'),
        'aX': coords.get('aX'),
        'vX0': coords.get('vX0'),
        'vY0': coords.get('vY0'),
        'vZ0': coords.get('vZ0'),
        'x0': coords.get('x0'),
        'y0': coords.get('y0'),
        'z0': coords.get('z0'),
        'pX': coords.get('pX'),
        'pZ': coords.get('pZ'),
        'playId': event.get('playId'),
    }

In [17]:
def pitch_api_call(schedule: pd.DataFrame) -> pd.DataFrame:
    """Extract, Transform, Load (ETL) process for MLB pitch data.
    Requires a DataFrame with a 'gamePk' column containing game IDs."""

    all_pitches = []

    for gamePk in schedule['gamePk']:
        game_url = f'https://statsapi.mlb.com/api/v1.1/game/{gamePk}/feed/live'

        for attempt in range(3):
            try:
                data = requests.get(game_url, timeout=10).json()
                break
            except (requests.exceptions.ConnectTimeout,
                    requests.exceptions.ReadTimeout,
                    requests.exceptions.ConnectionError) as e:
                if attempt == 2:
                    print(f"Failed gamePk={gamePk} after 3 attempts: {e}")
                    data = None
                else:
                    time.sleep(2 ** attempt)

        if data is None:
            continue

        catcher_states = get_catcher_states(data)
        pre_pitch, post_play, post_outs, pre_score, final_score = get_runner_states(data)
        pre_count = get_count_states(data)
        pre_challenges, final_challenges = get_challenge_states(data)

        errors = validate_runner_states(data, pre_pitch, post_play, post_outs)
        errors += validate_count_states(data, pre_count)
        errors += validate_challenge_states(data, final_challenges)
        errors += validate_score_states(data, final_score)
        errors += validate_catcher_states(data, catcher_states)
        if errors:
            print(f"gamePk={gamePk}: {len(errors)} validation errors")
            for e in errors[:5]:
                print('  ', e)

        for play in data['liveData']['plays']['allPlays']:
            for event in play['playEvents']:
                if event['type'] == 'pitch':
                    all_pitches.append(extract_pitch(event, play, data, pre_pitch, pre_count, post_outs, pre_challenges, pre_score, catcher_states))

        time.sleep(1)

    return pd.DataFrame(all_pitches)

In [19]:
pitches = pitch_api_call(schedule)

In [23]:
pitches = (
    pitches
    .pipe(drop_incomplete_abs)
    .pipe(midpoint)
    .pipe(signed_edge_dist)
    .pipe(original_call)
    .pipe(abs_classifier)
)

pitches = pitches.drop_duplicates(['gamePk', 'atBatIndex','eventIndex'])
pitches.to_parquet('pitch_data.parquet', engine='pyarrow', index=False)

Dropped 0 incomplete at bats (0 pitches removed)
